## Bronze to Silver: Data Cleaning and Transformation for Dimension Tables

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F
catalog_name = 'ecommerce'

###Brands

In [0]:
df_bronze = spark.table(f'{catalog_name}.bronze.brz_brands')
display(df_bronze.limit(10))

In [0]:
#Trimming the column brand_name
df_silver=df_bronze.withColumn('brand_name',F.trim(F.col('brand_name')))
df_silver.show(10)

In [0]:
#Removing special characters
df_silver = df_silver.withColumn("brand_code", F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]', ''))
df_silver.show(10)

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
# Anomalies dictionary
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

# PySpark replace
df_silver = df_silver.replace(to_replace=anomalies, subset=["category_code"])
df_silver.select("category_code").distinct().show()

In [0]:
# Writing transformed data to the silver layer (catalog: ecommerce, schema: silver, table: slv_brands)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

###Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
display(df_bronze.limit(10))

In [0]:
#Showing duplicate rows in the Column
df_duplicates = df_bronze.groupBy("category_code").count().filter(F.col("count") > 1)
display(df_duplicates)

In [0]:
#Droping duplicate rows
df_silver = df_bronze.dropDuplicates(['category_code'])
display(df_silver)

In [0]:
#Converting Column to UpperCase
df_silver = df_silver.withColumn("category_code", F.upper(F.col("category_code")))
display(df_silver)

In [0]:
# Write transformed data to the silver layer (catalog: ecommerce, schema: silver, table: slv_category)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

###Products

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_products)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_products")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

In [0]:
display(df_bronze.limit(5))

In [0]:
# Replace the 'g' character with '' in the column weight_grams. Casting to Integer
df_silver = df_bronze.withColumn("weight_grams",
    F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType())
)
df_silver.select("weight_grams").show(5, truncate=False)

In [0]:
# Replace the ',' with '.' and Casting to Float
df_silver = df_silver.withColumn(
    "length_cm",
    F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType())
)
df_silver.select("length_cm").show(5)

In [0]:
# Converting category_code and brand_code to upper case
df_silver = df_silver.withColumn(
    "category_code",
    F.upper(F.col("category_code"))
).withColumn(
    "brand_code",
    F.upper(F.col("brand_code"))
)
df_silver.select("category_code", "brand_code").show(5)

In [0]:
df_silver.select("material").distinct().show()

In [0]:
# Fixing gramatic mistakes
df_silver = df_silver.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Alumium", "Aluminum")
     .when(F.col("material") == "Ruber", "Rubber")
     .otherwise(F.col("material"))
)
df_silver.select("material").distinct().show()   

In [0]:
#Negative values in `rating_count`
df_silver.filter(F.col('rating_count')<0).select("rating_count").show(5)


In [0]:
# Convert negative rating_count to positive
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  # if null, replace with 0
)
df_silver.select("rating_count").show(5)

In [0]:
df_silver.select("width_cm","height_cm").show(5)

In [0]:
# Checking the final cleaned data

df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

In [0]:
# Writing transformed data to the silver layer (catalog: ecommerce, schema: silver, table: slv_dim_products)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

###Customers

In [0]:
# Reading the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")

# Getting row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

display(df_bronze.limit(10))

In [0]:
#Handling NULL values in Customer_id
null_count = df_bronze.filter(F.col("customer_id").isNull()).count()
print(f"Null count: {null_count}")
df_bronze.filter(F.col("customer_id").isNull()).show(5)

In [0]:
# Dropping rows where 'customer_id' is null
df_silver = df_bronze.dropna(subset=["customer_id"])

# Getting row count
row_count = df_silver.count()
print(f"Row count after droping null values: {row_count}")

In [0]:
#Handling NULL values in Phone
null_count = df_silver.filter(F.col("phone").isNull()).count()
print(f"Number of nulls in phone: {null_count}")
df_silver.filter(F.col("phone").isNull()).show(5)

In [0]:
### Fill null values with 'Not Available'
df_silver = df_silver.fillna("Not Available", subset=["phone"])

# sanity check (If any nulls still exist)
df_silver.filter(F.col("phone").isNull()).show()

In [0]:
#Verifying other columns data quality
df_silver.select("country","country_code").distinct().show()

In [0]:
# Writing transformed data to the silver layer (catalog: ecommerce, schema: silver, table: slv_customers)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

###Calendar

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

display(df_bronze.limit(10))

In [0]:
df_bronze.printSchema()

In [0]:
from pyspark.sql.functions import to_date

# Converting the string column to a date type
df_silver = df_bronze.withColumn("date", to_date(df_bronze["date"], "dd-MM-yyyy"))
df_silver.printSchema()
df_silver.show(5)

In [0]:
# Finding duplicate rows in the DataFrame
duplicates = df_silver.groupBy('date').count().filter("count > 1")

# Showing the duplicate rows
print("Total duplicated Rows: ", duplicates.count())
duplicates.show()

In [0]:
# Removing duplicate rows
df_silver = df_silver.dropDuplicates(['date'])
# Getting row count
row_count = df_silver.count()

print("Rows After removing Duplicates: ", row_count)

In [0]:
# Capitalize first letter of each word in day_name
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))

df_silver.select("day_name").distinct().show()

In [0]:
#Converting negative `week_of_year` to positive
df_silver = df_silver.withColumn("week_of_year", F.abs(F.col("week_of_year")))

df_silver.select("week_of_year").show(10)

In [0]:
#Enhancing `quarter` and `week_of_year` column
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))

df_silver.show(5)

In [0]:
# Renaming columns
df_silver = df_silver.withColumnRenamed("day_name", "weekday")
df_silver = df_silver.withColumnRenamed("week_of_year", "week")
#Verifying data
df_silver.show(10)

In [0]:
# Write transformed data to the silver layer (catalog: ecommerce, schema: silver, table: slv_calendar)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")